# 03 Silver

Lee lo nuevo en bronze con stream de Delta y checkpoint propio. En cada batch
quita duplicados de `(MMSI, BaseDateTime)` y separa filas válidas de cuarentena.
Las válidas entran a silver con un `MERGE` que solo mira los `fecha_dia` de ese
batch. Silver está agrupada con `CLUSTER BY (fecha_dia)` para podar esos días.
Cuarentena es append. `fecha` solo etiqueta la métrica de la ejecución; no filtra
el stream.

No se ejecuta en paralelo con otra sesión.

## Reglas de calidad

Esta Parte 2 no copia la Parte 1. La conclusión de la Parte 1 pedía **poner el
campo a null y conservar la fila** cuando el valor es imposible o es un centinela:
`SOG = 102.3`, `SOG > 60`, `COG = 360`, `Heading = 511`, Heading fuera de 0-359,
`Length`/`Width <= 0` y `Status = 15`. Aquí, si SOG o COG no son confiables, la
fila entera va a cuarentena.

Silver es lo que lee gold: trayectorias y distancias (haversine sobre `LAT`/`LON`),
presencia en puerto, matriz de tráfico y la velocidad media `sog_media`. Gold no
vuelve a filtrar. Un `SOG` de 102.3 nudos inflaría esa media, como ya midió la
Parte 1. Por eso la fila no se queda en silver con el campo anulado: pasa a
cuarentena con `motivo`. El registro no se pierde, se puede auditar y el conteo
entra en la métrica de la ejecución.

`Heading = 511` se queda, con su valor. Es rumbo ausente (norma ITU; en la Parte 1
era más de la mitad de la semana), no una posición mala. Un Heading fuera de 0-359
también se queda. `Length`/`Width <= 0` y `Status = 15` la Parte 1
los ponía a null. `Draft`, `IMO`, `Cargo` y `CallSign` pedía conservarlos sin
usarlos como llave. Aquí ninguno de esos campos mueve la fila ni se reescribe:
gold arma la posición con `LAT`, `LON` y el tiempo.

La Parte 1 también eliminaba outliers fuera de la caja de estudio (`LAT` 0-72,
`LON` -180 a -60). Aquí solo cuentan el rango global y el cero.

### Limpieza en el batch

| Regla | Condición | Por qué |
|---|---|---|
| Deduplicación en el batch | `dropDuplicates` de `(MMSI, BaseDateTime)` antes de marcar `motivo` | Deja una fila por buque e instante dentro del batch. Cubre el duplicado exacto de la Parte 1 `(MMSI, BaseDateTime, LAT, LON)` y el mismo instante en dos lugares. El `MERGE` no reinserta una clave que ya esté en los `fecha_dia` de ese batch. |

### Cuarentena

Una fila puede fallar varias reglas. `motivo` las junta con coma. Si `motivo`
queda vacío, la fila es válida y entra a silver.

| motivo | Condición | Por qué |
|---|---|---|
| `mmsi_invalido` | `MMSI` nulo, longitud distinta de 9, o solo ceros | Sin buque no hay trayectoria ni agregación. La Parte 1 también eliminaba estas filas, incluido `MMSI = 0`. |
| `sin_timestamp` | `BaseDateTime` nulo | Sin instante no hay `fecha_dia`, ni orden de la trayectoria, ni clave del `MERGE`. |
| `latitud_invalida` | `LAT` nula, menor que -90, mayor que 90, o exactamente 0 | No es una posición para haversine ni para H3. La Parte 1 marcaba el mismo rango global y el 0.0. |
| `longitud_invalida` | `LON` nula, menor que -180, mayor que 180, o exactamente 0 | Igual que la latitud. |
| `sog_invalido` | `SOG` nulo, menor que 0, o mayor que 60 | Incluye el centinela ITU `102.3`, porque 102.3 es mayor que 60. La Parte 1 anulaba `SOG = 102.3` y `SOG > 60` y conservaba la fila. Aquí la fila va a cuarentena. |
| `cog_invalido` | `COG` nulo, menor que 0, o mayor que 360 | Rumbo sobre el fondo fuera de rango. La Parte 1 anulaba el centinela `COG = 360` y conservaba la fila. 360 no es mayor que 360, así que `COG = 360` se queda en silver. |


In [ ]:
%run ./_config


In [ ]:
from pyspark.sql import functions as F

COLUMNAS = [
    "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
    "VesselName", "IMO", "CallSign", "VesselType", "Status",
    "Length", "Width", "Draft", "Cargo", "TransceiverClass",
    "_source_file", "_ingested_at",
]


def con_motivo(df):
    """Marca las reglas fallidas. concat_ws omite los null, asi que '' significa que la fila es valida."""
    mmsi_mal = F.col("MMSI").isNull() | (F.length("MMSI") != 9) | F.col("MMSI").rlike("^0+$")
    ts_mal = F.col("BaseDateTime").isNull()
    lat_mal = F.col("LAT").isNull() | (F.col("LAT") < -90) | (F.col("LAT") > 90) | (F.col("LAT") == 0)
    lon_mal = F.col("LON").isNull() | (F.col("LON") < -180) | (F.col("LON") > 180) | (F.col("LON") == 0)
    sog_mal = F.col("SOG").isNull() | (F.col("SOG") < 0) | (F.col("SOG") > 60)
    cog_mal = F.col("COG").isNull() | (F.col("COG") < 0) | (F.col("COG") > 360)
    motivo = F.concat_ws(
        ",",
        F.when(mmsi_mal, F.lit("mmsi_invalido")),
        F.when(ts_mal, F.lit("sin_timestamp")),
        F.when(lat_mal, F.lit("latitud_invalida")),
        F.when(lon_mal, F.lit("longitud_invalida")),
        F.when(sog_mal, F.lit("sog_invalido")),
        F.when(cog_mal, F.lit("cog_invalido")),
    )
    return df.withColumn("motivo", motivo)


def escribir_silver(sesion, validos) -> None:
    fechas = [
        fila.fecha_dia.isoformat()
        for fila in validos.select("fecha_dia").distinct().collect()
        if fila.fecha_dia is not None
    ]
    if not fechas:
        return
    lista = ", ".join(f"DATE '{dia}'" for dia in fechas)
    vista = "oceanwatch_silver_batch"
    validos.drop("motivo").createOrReplaceTempView(vista)
    sesion.sql(f"""
        MERGE INTO {SILVER} AS destino
        USING {vista} AS origen
        ON destino.fecha_dia IN ({lista})
       AND destino.MMSI = origen.MMSI
       AND destino.BaseDateTime = origen.BaseDateTime
        WHEN NOT MATCHED THEN INSERT *
    """)
    sesion.catalog.dropTempView(vista)


def escribir_cuarentena(invalidos) -> None:
    (
        invalidos.select(*COLUMNAS, "motivo")
        .write.format("delta")
        .mode("append")
        .saveAsTable(CUARENTENA)
    )


def aplicar_silver(batch_df, batch_id: int) -> None:
    sesion = batch_df.sparkSession
    n_entrada = batch_df.count()
    marcado = (
        con_motivo(batch_df.dropDuplicates(["MMSI", "BaseDateTime"]))
        .withColumn("fecha_dia", F.to_date("BaseDateTime"))
    )
    validos = marcado.where(F.col("motivo") == "")
    invalidos = marcado.where(F.col("motivo") != "")
    n_validas = validos.count()
    n_cuarentena = invalidos.count()
    if n_validas:
        escribir_silver(sesion, validos)
    if n_cuarentena:
        escribir_cuarentena(invalidos)
    sesion.sql(f"""
        INSERT INTO {METRICAS}
        SELECT
          DATE '{fecha_dia.isoformat()}',
          'silver',
          CAST(NULL AS BIGINT),
          CAST(NULL AS BIGINT),
          CAST({n_entrada} AS BIGINT),
          CAST({n_validas} AS BIGINT),
          CAST({n_cuarentena} AS BIGINT),
          current_timestamp()
    """)
    print(f"batch {batch_id}: entrada={n_entrada} validas={n_validas} cuarentena={n_cuarentena}")


for consulta in list(spark.streams.active):
    if consulta.name == "oceanwatch_silver_ais":
        consulta.stop()

consulta = (
    spark.readStream.format("delta")
    .table(BRONZE)
    .writeStream.foreachBatch(aplicar_silver)
    .queryName("oceanwatch_silver_ais")
    .option("checkpointLocation", CHECKPOINT_SILVER)
    .trigger(availableNow=True)
    .start()
)
consulta.awaitTermination()
print(f"Ejecución silver {fecha_dia.isoformat()} terminada.")
